# CONFIGURATION - TRAINING

In [ ]:
# cVAE Hyperparameters
LATENT_DIM = 10          # Size of latent space
HIDDEN_DIM = 128         # Size of hidden layers
LEARNING_RATE = 1e-3     # Learning rate
BATCH_SIZE = 64          # Batch size
NUM_EPOCHS = 100         # Maximum number of epochs
BETA = 1.0               # Weight for KL divergence (β-VAE)

# Early stopping
PATIENCE = 15            # Stop if no improvement for this many epochs

# Paths
DATA_PATH = '/content/drive/MyDrive/NARIT/Project2_Syntheticdata/Dataset/prepared_data_80_20.pkl'
MODEL_SAVE_PATH = '/content/drive/MyDrive/NARIT/Project2_Syntheticdata/Models/best_cvae_model.pth'
SYNTHETIC_DATA_BASE_PATH = '/content/drive/MyDrive/NARIT/Project2_Syntheticdata/Dataset/'  # Base directory for synthetic data

# STEP 1: Load Prepared Data

In [ ]:
print("\n📂 Loading prepared data...")
loaded_data = joblib.load(DATA_PATH)

X_train = loaded_data['X_train']
X_val = loaded_data['X_val']
y_train = loaded_data['y_train']
y_val = loaded_data['y_val']
label_encoder = loaded_data['label_encoder']
scaler = loaded_data['scaler']
num_features = loaded_data['num_features']
num_classes = loaded_data['num_classes']

print(f"✓ Data loaded successfully!")
print(f"  Train samples: {X_train.shape[0]}")
print(f"  Val samples: {X_val.shape[0]}")
print(f"  Features: {num_features}")
print(f"  Classes: {num_classes}")
print(f"  Class names: {list(label_encoder.classes_)}")

# STEP 2: Setup Device

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n💻 Using device: {device}")

In [ ]:
if device == "cuda":
  torch.cuda.empty_cache()
  print("✓ GPU cache cleared")
else:
  print("You are using CPU.")

# STEP 3: Create PyTorch Dataset and DataLoader

In [ ]:
class LightCurveDataset(Dataset):
    """Custom dataset for light curve features with labels"""
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# Create datasets
train_dataset = LightCurveDataset(X_train, y_train)
val_dataset = LightCurveDataset(X_val, y_val)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\n✓ DataLoaders created:")
print(f"  Training batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")

# STEP 4: Define cVAE Architecture

In [ ]:
class Encoder(nn.Module):
    """Encoder: Takes features + condition → outputs mean and log_variance"""
    def __init__(self, input_dim, condition_dim, hidden_dim, latent_dim):
        super(Encoder, self).__init__()

        self.fc1 = nn.Linear(input_dim + condition_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc_mean = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x, c):
        xc = torch.cat([x, c], dim=1)
        h = F.relu(self.fc1(xc))
        h = F.relu(self.fc2(h))
        mean = self.fc_mean(h)
        logvar = self.fc_logvar(h)
        return mean, logvar


class Decoder(nn.Module):
    """Decoder: Takes latent sample + condition → reconstructs features"""
    def __init__(self, latent_dim, condition_dim, hidden_dim, output_dim):
        super(Decoder, self).__init__()

        self.fc1 = nn.Linear(latent_dim + condition_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)

    def forward(self, z, c):
        zc = torch.cat([z, c], dim=1)
        h = F.relu(self.fc1(zc))
        h = F.relu(self.fc2(h))
        x_recon = self.fc3(h)
        return x_recon


class ConditionalVAE(nn.Module):
    """Conditional VAE with separate encoder/decoder and generation method"""
    def __init__(self, input_dim, num_classes, latent_dim, hidden_dim):
        super(ConditionalVAE, self).__init__()

        self.latent_dim = latent_dim
        self.num_classes = num_classes

        # Encoder and Decoder
        self.encoder = Encoder(input_dim, num_classes, hidden_dim, latent_dim)
        self.decoder = Decoder(latent_dim, num_classes, hidden_dim, input_dim)

    def reparameterize(self, mean, logvar):
        """Reparameterization trick: z = mean + std * epsilon"""
        std = torch.exp(0.5 * logvar)
        epsilon = torch.randn_like(std)
        return mean + std * epsilon

    def forward(self, x, c):
        """Forward pass: encode → reparameterize → decode"""
        # Encode
        mean, logvar = self.encoder(x, c)

        # Reparameterize
        z = self.reparameterize(mean, logvar)

        # Decode
        x_recon = self.decoder(z, c)

        return x_recon, mean, logvar

    def generate(self, num_samples, condition, device):
        """Generate new samples for a given condition (class)"""
        self.eval()

        with torch.no_grad():
            # Sample from standard normal distribution
            z = torch.randn(num_samples, self.latent_dim).to(device)

            # Create condition vector (one-hot)
            c = torch.zeros(num_samples, self.num_classes).to(device)
            c[:, condition] = 1

            # Decode
            generated = self.decoder(z, c)

        return generated.cpu().numpy()

print("✓ cVAE architecture defined")

# STEP 5: Initialize Model

In [ ]:
print(f"\n🔧 Initializing model...")
cvae_model = ConditionalVAE(
    input_dim=num_features,
    num_classes=num_classes,
    latent_dim=LATENT_DIM,
    hidden_dim=HIDDEN_DIM
).to(device)

print(f"✓ Model created with {sum(p.numel() for p in cvae_model.parameters()):,} parameters")

# Optimizer
optimizer = optim.Adam(cvae_model.parameters(), lr=LEARNING_RATE)
print(f"✓ Optimizer: Adam with lr={LEARNING_RATE}")

# STEP 6: Define Loss Function

In [ ]:
def cvae_loss(x_recon, x, mean, logvar, beta=1.0):
    """
    Conditional VAE Loss = Reconstruction Loss + β * KL Divergence

    Args:
        x_recon: Reconstructed features
        x: Original features
        mean: Latent space mean
        logvar: Latent space log variance
        beta: Weight for KL divergence (β-VAE)
    """
    # Reconstruction loss (MSE)
    recon_loss = F.mse_loss(x_recon, x, reduction='mean')

    # KL divergence: KL(q(z|x,c) || p(z))
    # -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    kl_loss = -0.5 * torch.mean(1 + logvar - mean.pow(2) - logvar.exp())

    # Total loss
    total_loss = recon_loss + beta * kl_loss

    return total_loss, recon_loss, kl_loss

print("✓ Loss function defined")

# STEP 7: Training Loop (RUN ONCE)

In [ ]:
print("\n🚀 STARTING TRAINING")
print("="*80)
print(f"Configuration:")
print(f"  Latent dim: {LATENT_DIM}")
print(f"  Hidden dim: {HIDDEN_DIM}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Max epochs: {NUM_EPOCHS}")
print(f"  Beta (KL weight): {BETA}")
print(f"  Early stopping patience: {PATIENCE}")
print("="*80)

# Training metrics tracking
training_metrics = {
    'train_losses': [],
    'val_losses': [],
    'best_val_loss': float('inf'),
    'best_epoch': 0,
    'best_train_total_loss': 0,
    'best_train_recon_loss': 0,
    'best_train_kl_loss': 0,
    'best_val_total_loss': 0,
    'best_val_recon_loss': 0,
    'best_val_kl_loss': 0
}

patience_counter = 0
start_time = time.time()

# ============================================================================
# TRAINING LOOP
# ============================================================================
for epoch in range(1, NUM_EPOCHS + 1):
    # ========================================================================
    # Training Phase
    # ========================================================================
    cvae_model.train()
    train_total_loss = 0
    train_recon_loss = 0
    train_kl_loss = 0

    for batch_idx, (features, labels) in enumerate(train_loader):
        features = features.to(device)
        labels = labels.to(device)

        # Create one-hot encoded condition
        condition = torch.zeros(features.size(0), num_classes).to(device)
        condition.scatter_(1, labels.unsqueeze(1), 1)

        # Forward pass
        x_recon, mean, logvar = cvae_model(features, condition)

        # Calculate loss
        total_loss, recon_loss, kl_loss = cvae_loss(
            x_recon, features, mean, logvar, beta=BETA
        )

        # Backward pass
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        # Accumulate losses
        train_total_loss += total_loss.item()
        train_recon_loss += recon_loss.item()
        train_kl_loss += kl_loss.item()

    # Average training losses
    train_total_loss /= len(train_loader)
    train_recon_loss /= len(train_loader)
    train_kl_loss /= len(train_loader)

    # ========================================================================
    # Validation Phase
    # ========================================================================
    cvae_model.eval()
    val_total_loss = 0
    val_recon_loss = 0
    val_kl_loss = 0

    with torch.no_grad():
        for features, labels in val_loader:
            features = features.to(device)
            labels = labels.to(device)

            # Create one-hot encoded condition
            condition = torch.zeros(features.size(0), num_classes).to(device)
            condition.scatter_(1, labels.unsqueeze(1), 1)

            # Forward pass
            x_recon, mean, logvar = cvae_model(features, condition)

            # Calculate loss
            total_loss, recon_loss, kl_loss = cvae_loss(
                x_recon, features, mean, logvar, beta=BETA
            )

            # Accumulate losses
            val_total_loss += total_loss.item()
            val_recon_loss += recon_loss.item()
            val_kl_loss += kl_loss.item()

    # Average validation losses
    val_total_loss /= len(val_loader)
    val_recon_loss /= len(val_loader)
    val_kl_loss /= len(val_loader)

    # ========================================================================
    # Logging
    # ========================================================================
    training_metrics['train_losses'].append(train_total_loss)
    training_metrics['val_losses'].append(val_total_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f"\nEpoch [{epoch:3d}/{NUM_EPOCHS}]")
        print(f"  Train - Total: {train_total_loss:.4f}, "
              f"Recon: {train_recon_loss:.4f}, KL: {train_kl_loss:.4f}")
        print(f"  Val   - Total: {val_total_loss:.4f}, "
              f"Recon: {val_recon_loss:.4f}, KL: {val_kl_loss:.4f}")

    # ========================================================================
    # Model Checkpoint (Save best model)
    # ========================================================================
    if val_total_loss < training_metrics['best_val_loss']:
        training_metrics['best_val_loss'] = val_total_loss
        training_metrics['best_epoch'] = epoch
        training_metrics['best_train_total_loss'] = train_total_loss
        training_metrics['best_train_recon_loss'] = train_recon_loss
        training_metrics['best_train_kl_loss'] = train_kl_loss
        training_metrics['best_val_total_loss'] = val_total_loss
        training_metrics['best_val_recon_loss'] = val_recon_loss
        training_metrics['best_val_kl_loss'] = val_kl_loss

        # Save model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_total_loss,
            'val_loss': val_total_loss,
            'config': {
                'latent_dim': LATENT_DIM,
                'hidden_dim': HIDDEN_DIM,
                'num_features': num_features,
                'num_classes': num_classes,
                'learning_rate': LEARNING_RATE,
                'beta': BETA
            }
        }, MODEL_SAVE_PATH)

        print(f"  💾 New best model saved (epoch {epoch})")
        patience_counter = 0
    else:
        patience_counter += 1

    # ========================================================================
    # Early Stopping
    # ========================================================================
    if patience_counter >= PATIENCE:
        print(f"\n⏹️ Early stopping at epoch {epoch}")
        print(f"   No improvement for {PATIENCE} epochs")
        break

# Calculate total training time
training_time = time.time() - start_time
training_metrics['training_time'] = training_time
training_metrics['total_epochs_trained'] = epoch

print(f"\n" + "="*80)
print("✅ TRAINING COMPLETE!")
print("="*80)
print(f"Training time: {training_time/60:.2f} minutes")
print(f"Total epochs: {epoch}")
print(f"Best epoch: {training_metrics['best_epoch']}")
print(f"\nBest Epoch Metrics (Epoch {training_metrics['best_epoch']}):")
print(f"  Train Loss - Total: {training_metrics['best_train_total_loss']:.4f}, "
      f"Recon: {training_metrics['best_train_recon_loss']:.4f}, "
      f"KL: {training_metrics['best_train_kl_loss']:.4f}")
print(f"  Val Loss   - Total: {training_metrics['best_val_total_loss']:.4f}, "
      f"Recon: {training_metrics['best_val_recon_loss']:.4f}, "
      f"KL: {training_metrics['best_val_kl_loss']:.4f}")
print("="*80)
print(f"\n📌 Model saved at: {MODEL_SAVE_PATH}")
print(f"\n🎉 You can now use the GENERATION sections below to create synthetic data!")

---
# 🎨 DATA GENERATION SECTION
## Run this section as many times as you want with different classes!
---

## STEP 8: Load Best Model (Run once after training)

In [ ]:
print(f"\n📥 Loading best model for generation...")

# Load checkpoint to get configuration
checkpoint = torch.load(MODEL_SAVE_PATH)

# Reinitialize the model with saved configuration
# This prevents conflicts if 'model' variable was overwritten by XGBoost or other models
cvae_model = ConditionalVAE(
    input_dim=num_features,
    num_classes=num_classes,
    latent_dim=checkpoint['config']['latent_dim'],
    hidden_dim=checkpoint['config']['hidden_dim']
).to(device)

# Load the trained weights
cvae_model.load_state_dict(checkpoint['model_state_dict'])
cvae_model.eval()  # Set to evaluation mode

print(f"✓ Best model loaded (from epoch {checkpoint['epoch']})")
print(f"✓ Model configuration:")
print(f"  Latent dim: {checkpoint['config']['latent_dim']}")
print(f"  Hidden dim: {checkpoint['config']['hidden_dim']}")
print(f"\n📋 Available classes for generation:")
for i, class_name in enumerate(label_encoder.classes_):
    print(f"  {i}: {class_name}")

## STEP 9: Configure Generation Parameters (CHANGE THESE AS NEEDED)

In [ ]:
# ⬇️⬇️⬇️ CHANGE THESE VALUES TO GENERATE DIFFERENT DATA ⬇️⬇️⬇️

TARGET_CLASS = "var_Cataclysmic"  # Which class to generate
NUM_SAMPLES_TO_GENERATE = 2500    # How many samples to generate

# Optional: Set custom filename (leave as None for auto-generated name)
CUSTOM_FILENAME = None  # e.g., "my_synthetic_data.pkl" or None

# ⬆️⬆️⬆️ CHANGE THESE VALUES TO GENERATE DIFFERENT DATA ⬆️⬆️⬆️

# Verify target class exists
if TARGET_CLASS not in label_encoder.classes_:
    raise ValueError(f"Target class '{TARGET_CLASS}' not found in classes: {list(label_encoder.classes_)}")

target_class_index = label_encoder.transform([TARGET_CLASS])[0]

print(f"\n🎯 Generation Configuration:")
print(f"  Target class: '{TARGET_CLASS}' (index: {target_class_index})")
print(f"  Number of samples: {NUM_SAMPLES_TO_GENERATE:,}")
if CUSTOM_FILENAME:
    print(f"  Custom filename: {CUSTOM_FILENAME}")
else:
    print(f"  Filename: Auto-generated")

## STEP 10: Generate Synthetic Data

In [ ]:
print(f"\n🎨 GENERATING SYNTHETIC DATA")
print("="*80)
print(f"Target class: '{TARGET_CLASS}' (index: {target_class_index})")
print(f"Number of samples: {NUM_SAMPLES_TO_GENERATE:,}")
print("="*80)

generation_start = time.time()

# Generate data
synthetic_features_scaled = cvae_model.generate(
    num_samples=NUM_SAMPLES_TO_GENERATE,
    condition=target_class_index,
    device=device
)

generation_time = time.time() - generation_start

print(f"\n✓ Generated {NUM_SAMPLES_TO_GENERATE:,} samples in {generation_time:.2f} seconds")
print(f"  Shape: {synthetic_features_scaled.shape}")
print(f"  Data is in STANDARDIZED form (same scale as training data)")

# Create labels for synthetic data
synthetic_labels = np.array([target_class_index] * NUM_SAMPLES_TO_GENERATE)

# Display statistics
print(f"\n📊 Generated Data Statistics:")
print(f"  Mean: {synthetic_features_scaled.mean():.6f}")
print(f"  Std: {synthetic_features_scaled.std():.6f}")
print(f"  Min: {synthetic_features_scaled.min():.6f}")
print(f"  Max: {synthetic_features_scaled.max():.6f}")

## STEP 11: Save Generated Data

In [ ]:
# Determine filename
if CUSTOM_FILENAME:
    # Use custom filename
    filename = CUSTOM_FILENAME
    if not filename.endswith('.pkl'):
        filename += '.pkl'
else:
    # Auto-generate filename based on class and number of samples
    safe_class_name = TARGET_CLASS.replace('/', '_').replace(' ', '_')
    filename = f"synthetic_{safe_class_name}_{NUM_SAMPLES_TO_GENERATE}.pkl"

SYNTHETIC_DATA_PATH = SYNTHETIC_DATA_BASE_PATH + filename

print(f"\n💾 Saving synthetic data...")
print(f"  Filename: {filename}")

# Package synthetic data
synthetic_data_package = {
    'features_scaled': synthetic_features_scaled,  # Standardized features
    'labels_encoded': synthetic_labels,            # Encoded labels (integers)
    'target_class': TARGET_CLASS,                  # Class name
    'target_class_index': target_class_index,      # Class index
    'num_samples': NUM_SAMPLES_TO_GENERATE,
    'feature_names': loaded_data.get('feature_names', None),
    'scaler': scaler,  # Include scaler for potential inverse transform
    'label_encoder': label_encoder,
    'generation_time': generation_time,
    'training_metrics': training_metrics,
    'model_config': {
        'latent_dim': LATENT_DIM,
        'hidden_dim': HIDDEN_DIM,
        'learning_rate': LEARNING_RATE,
        'batch_size': BATCH_SIZE,
        'beta': BETA
    }
}

# Save to pickle file
joblib.dump(synthetic_data_package, SYNTHETIC_DATA_PATH)

print(f"✓ Synthetic data saved to:")
print(f"  {SYNTHETIC_DATA_PATH}")
print(f"\n🎉 Done! You can now:")
print(f"  1. Change TARGET_CLASS, NUM_SAMPLES_TO_GENERATE, or CUSTOM_FILENAME in Step 9")
print(f"  2. Re-run Steps 9-11 to generate more data for different classes")
print(f"  3. No need to retrain the model!")

## (Optional) Quick Generation Helper Function

In [ ]:
def generate_and_save(class_name, num_samples, custom_filename=None):
    """
    Helper function to quickly generate and save synthetic data.
    
    Usage:
        generate_and_save("var_Cataclysmic", 2500)
        generate_and_save("var_RRLyr", 1000)
        generate_and_save("var_Mira", 3000, custom_filename="mira_large.pkl")
    """
    # Verify class exists
    if class_name not in label_encoder.classes_:
        print(f"❌ Error: Class '{class_name}' not found!")
        print(f"Available classes: {list(label_encoder.classes_)}")
        return
    
    class_index = label_encoder.transform([class_name])[0]
    
    print(f"\n🎨 Generating {num_samples:,} samples for '{class_name}'...")
    
    # Generate
    start_time = time.time()
    features = cvae_model.generate(
        num_samples=num_samples,
        condition=class_index,
        device=device
    )
    gen_time = time.time() - start_time
    
    labels = np.array([class_index] * num_samples)
    
    # Determine filename
    if custom_filename:
        filename = custom_filename
        if not filename.endswith('.pkl'):
            filename += '.pkl'
    else:
        safe_name = class_name.replace('/', '_').replace(' ', '_')
        filename = f"synthetic_{safe_name}_{num_samples}.pkl"
    
    filepath = SYNTHETIC_DATA_BASE_PATH + filename
    
    data_package = {
        'features_scaled': features,
        'labels_encoded': labels,
        'target_class': class_name,
        'target_class_index': class_index,
        'num_samples': num_samples,
        'feature_names': loaded_data.get('feature_names', None),
        'scaler': scaler,
        'label_encoder': label_encoder,
        'generation_time': gen_time,
        'model_config': {
            'latent_dim': LATENT_DIM,
            'hidden_dim': HIDDEN_DIM,
            'beta': BETA
        }
    }
    
    joblib.dump(data_package, filepath)
    
    print(f"✅ Generated in {gen_time:.2f}s and saved to: {filename}")
    return filepath

print("✓ Helper function defined!")
print("\nUsage examples:")
print("  generate_and_save('var_Cataclysmic', 2500)")
print("  generate_and_save('var_RRLyr', 1000)")
print("  generate_and_save('var_Mira', 3000, custom_filename='mira_large.pkl')")

## (Optional) Batch Generation Example

In [ ]:
# Example: Generate data for multiple classes at once
# Uncomment and modify as needed:

# generation_plan = [
#     ("var_Cataclysmic", 2500),
#     ("var_RRLyr", 1000),
#     ("var_Mira", 1500),
#     ("var_EB", 2000),
# ]

# print("🚀 Starting batch generation...\n")
# for class_name, num_samples in generation_plan:
#     generate_and_save(class_name, num_samples)
#     print()  # Add spacing between generations

# print("✅ Batch generation complete!")

print("💡 Tip: Uncomment the code above to generate multiple classes at once!")